## Streszczenie

Celem projektu jest wytrenowanie dwóch modeli, generujących muzykę, korzytsając z bibliotek torch. Do trenowania modeli LSTM oraz transformerowego użyta została oprawa muzyczna gier DOOM oraz DOOM 2 zapisanych w formacie mid. Odrzucono powtarzające się i odstające od reszty utwory a następnie utworzeno na ich podstawie słownik tokenów. Trening odbywał sie w wielu wariantach, aby umożliwić modelom jak najlepsze odwzorowanie klasycznej oprawy muzycznej gier z serii DOOM.

## 2. Wstęp / wprowadzenie

Generowanie muzyki z użyciem sieci neuronowych stanowi wymagające zadanie z zakresu modelowania sekwencji, ponieważ model musi uchwycić zarówno lokalne zależności między kolejnymi zdarzeniami, jak i dłuższą strukturę rytmiczną oraz motywiczną utworu. W przypadku muzyki symbolicznej problem ten rozpatruje się najczęściej na poziomie zdarzeń MIDI, co pozwala operować na reprezentacji nut, instrumentów i przesunięć czasowych zamiast na surowym sygnale audio.

W literaturze stosuje się w tym celu różne architektury autoregresyjne, w szczególności modele rekurencyjne typu LSTM oraz nowsze modele oparte na mechanizmie self-attention, takie jak Transformer. Pierwsze z nich dobrze modelują lokalną ciągłość sekwencji, natomiast drugie lepiej radzą sobie z uchwyceniem zależności długoterminowych i bardziej złożonej struktury muzycznej.

W projekcie rozważono oba podejścia w zadaniu generowania muzyki w stylistyce ścieżki dźwiękowej gier *Doom* i *Doom II* autorstwa Bobby'ego Prince'a. Wybrany zbiór danych jest stosunkowo mały, ale jednocześnie spójny stylistycznie i dostępny w formacie MIDI, co czyni go dobrym materiałem do eksperymentów z modelowaniem muzyki symbolicznej.

## 3. Opis kontekstu i celu

Projekt wpisuje się w obszar generowania muzyki symbolicznej, w którym utwór reprezentowany jest jako sekwencja zdarzeń MIDI przewidywanych na podstawie wcześniejszego kontekstu. Jest to zadanie zbliżone formalnie do modelowania języka, jednak w praktyce wymaga uchwycenia nie tylko kolejności zdarzeń, ale również regularności rytmicznej, powtarzalności motywów i spójności w dłuższych odcinkach sekwencji.

Istotnym kontekstem projektu jest porównanie dwóch klas architektur używanych w generowaniu muzyki: modeli rekurencyjnych LSTM oraz modeli opartych na mechanizmie uwagi, takich jak Transformer. LSTM stanowi klasyczne podejście do modelowania sekwencji i dobrze nadaje się do uchwycenia zależności lokalnych, natomiast Transformer lepiej radzi sobie z modelowaniem relacji długoterminowych dzięki bezpośredniemu dostępowi do całego kontekstu wejściowego.

Jako materiał treningowy wybrano utwory z gier *Doom* i *Doom II* autorstwa Bobby'ego Prince’a. Zbiór ten jest relatywnie mały, lecz jednorodny stylistycznie i dostępny w formacie MIDI.

Głównym celem projektu było zbudowanie i przetestowanie modeli zdolnych do generowania nowych sekwencji MIDI utrzymanych w stylistyce oryginalnej muzyki z serii *Doom*. Celem dodatkowym było sprawdzenie, jak model LSTM i model Transformer zachowują się w warunkach małego zbioru danych oraz które cechy generowanej muzyki każdy z nich odwzorowuje lepiej.

Od strony praktycznej projekt obejmował cały proces eksperymentalny: przygotowanie danych, tokenizację zdarzeń, implementację i trening modeli, a następnie generowanie i odsłuch otrzymanych sekwencji. Ocena jakości miała charakter zarówno ilościowy, na podstawie funkcji straty, jak i jakościowy, przez analizę odsłuchową wygenerowanej muzyki.

## 5. Opis danych

W projekcie wykorzystano zbiór plików MIDI zawierających muzykę z gier *Doom* i *Doom II*. Dane zostały pobrane z dwóch archiwów: [`doommusic.zip`](https://archive.org/download/doommusic) oraz [`doom2music.zip`](https://archive.org/download/doom2music), a po ich rozpakowaniu uzyskano łącznie 67 surowych plików MIDI.

Wybrany materiał ma charakter symboliczny, co oznacza, że każdy utwór zapisany jest jako zbiór zdarzeń muzycznych, takich jak nuty, instrumenty czy informacje czasowe, a nie jako sygnał audio. Taka forma danych jest wygodna w zadaniu generowania muzyki, ponieważ umożliwia analizę struktury utworu bez konieczności przetwarzania surowej fali dźwiękowej.

W ramach wstępnej analizy dla każdego pliku wyznaczono podstawowe statystyki, obejmujące między innymi czas trwania, liczbę instrumentów, liczbę nut, średnie tempo oraz informację o obecności ścieżki perkusyjnej. Na tej podstawie przeprowadzono czyszczenie zbioru: wykonano deduplikację po liczbie nut oraz odrzucono utwory krótsze niż 60 sekund. Usunięto też plik `dbunny.mid`, który nie był spójny stylistycznie z resztą danych. W rezultacie końcowy zbiór zawierał 42 utwory przeznaczone do dalszego przetwarzania.

Po przygotowaniu danych każdy utwór został zapisany jako sekwencja tokenów reprezentujących zdarzenia muzyczne. Łącznie otrzymano 382 035 tokenów dla 42 utworów, co wskazuje, że mimo niewielkiej liczby plików zbiór zawiera stosunkowo dużą liczbę elementów sekwencyjnych do uczenia modeli.

Dodatkowo wykonano prostą analizę eksploracyjną zbioru danych w celu lepszego zrozumienia jego struktury. Rozkład liczby nut w utworach pozwala ocenić zróżnicowanie gęstości muzycznej między ścieżkami, natomiast rozkład długości utworów pokazuje, że zbiór zawiera zarówno krótsze, jak i wyraźnie dłuższe kompozycje. Uzupełnieniem tej analizy jest wykres udziału utworów zawierających ścieżkę perkusyjną, co ma znaczenie z punktu widzenia charakterystyki brzmieniowej materiału.

![alt text](Note_count_distribution.png "Title")
**Rysunek 1.** Rozkład liczby nut w utworach.

![alt text](Track_len_dirtribution.png "Title")
**Rysunek 2.** Rozkład długości ścieżek MIDI w zbiorze danych.  

![alt text](contain_drums.png "Title")
**Rysunek 3.** Udział utworów zawierających ścieżkę perkusyjną.  